# A complete small river game

This notebook connects private deals, public cards, betting actions, information sets, showdown and exact expected value. The game is heads-up with one fixed bet size and no raises.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fractions import Fraction
import pandas as pd
from cards import hand_names, make_hand
from river_game import (
    BET, CALL, CHECK, FOLD,
    apply_action, evaluate_strategies, information_set_key,
    legal_actions, new_river_game, terminal_utility,
)

## 1. One river spot

The starting pot is 2 chips, both players have 10 chips behind, and the only bet size is 1 chip.

In [2]:
board = make_hand(['2c', '7d', '9h', 'Js', '3c'])
first_strong = make_hand(['Jc', 'Ad'])
first_weak = make_hand(['4d', '5d'])
second_strong = make_hand(['9c', 'Kd'])
second_weak = make_hand(['8c', 'Qd'])
root = new_river_game(board, first_strong, second_strong, pot=2, bet_size=1)

print('Board:', ' '.join(hand_names(board)))
print('Player 0:', ' '.join(hand_names(first_strong)))
print('Player 1:', ' '.join(hand_names(second_strong)))
print('Legal root actions:', legal_actions(root))
print('Game terminal:', root.terminal)

Board: 2c 3c 7d 9h Js
Player 0: Jc Ad
Player 1: 9c Kd
Legal root actions: ('check', 'bet')
Game terminal: False


The public board is already a river, so a `ChanceState` would have no card children. This `RiverGameState` is not terminal because betting remains.

## 2. Complete action tree

The possible terminal histories are check–check, bet–fold, bet–call, check–bet–fold and check–bet–call.

In [3]:
def terminal_rows(state):
    if state.terminal:
        return [{
            'history': ' → '.join(state.action_history),
            'pot': state.pot,
            'contributions': state.contributions,
            'player 0 utility': terminal_utility(state, 0),
            'player 1 utility': terminal_utility(state, 1),
        }]
    rows = []
    for action in legal_actions(state):
        rows.extend(terminal_rows(apply_action(state, action)))
    return rows

tree = pd.DataFrame(terminal_rows(root))
tree

,history,pot,contributions,player 0 utility,player 1 utility
0,check → check,2,"(0, 0)",2,0
1,check → bet → fold,3,"(0, 1)",0,2
2,check → bet → call,4,"(1, 1)",3,-1
3,bet → fold,3,"(1, 0)",2,0
4,bet → call,4,"(1, 1)",3,-1


## 3. Information sets hide the opponent's hand

These two underlying states give player 0 the same cards, board and public history but different opponent hands. Player 0 must receive the same information-set key.

In [4]:
deal_one = new_river_game(board, first_strong, second_strong)
deal_two = new_river_game(board, first_strong, second_weak)
key_one = information_set_key(deal_one)
key_two = information_set_key(deal_two)

print('Keys are equal:', key_one == key_two)
print('Observed own hand:', ' '.join(hand_names(key_one.own_hand)))
print('Observed history:', key_one.action_history)
print('Opponent hand is a key field:', 'opponent_hand' in key_one.__dataclass_fields__)

Keys are equal: True
Observed own hand: Jc Ad
Observed history: ()
Opponent hand is a key field: False


## 4. Two simple strategy profiles

Player 0 bets the pair of jacks and checks the weak hand. Player 1 always checks when checked to. We compare player 1 always calling a bet with always folding to it. Strategies see only an information-set key and the legal actions.

In [5]:
def deterministic(actions, choice):
    return {action: Fraction(action == choice) for action in actions}

def first_policy(key, actions):
    if actions == (CHECK, BET):
        choice = BET if key.own_hand == first_strong else CHECK
    else:
        choice = FOLD
    return deterministic(actions, choice)

def second_calls(key, actions):
    return deterministic(actions, CALL if actions == (FOLD, CALL) else CHECK)

def second_folds(key, actions):
    return deterministic(actions, FOLD if actions == (FOLD, CALL) else CHECK)

In [6]:
first_range = [first_strong, first_weak]
second_range = [second_strong, second_weak]
calls = evaluate_strategies(board, first_range, second_range, (first_policy, second_calls))
folds = evaluate_strategies(board, first_range, second_range, (first_policy, second_folds))

pd.DataFrame([
    {'player 1 response': 'always call', 'player 0 EV': calls.first_player_ev, 'player 1 EV': calls.second_player_ev},
    {'player 1 response': 'always fold', 'player 0 EV': folds.first_player_ev, 'player 1 EV': folds.second_player_ev},
])

,player 1 response,player 0 EV,player 1 EV
0,always call,3/2,1/2
1,always fold,1,1


## 5. Deal-level explanation

All four compatible deals have probability 1/4. The deal table makes the aggregate EV directly checkable.

In [7]:
pd.DataFrame([
    {
        'player 0 hand': ' '.join(hand_names(deal.first)),
        'player 1 hand': ' '.join(hand_names(deal.second)),
        'probability': str(deal.probability),
        'player 0 utility': deal.first_player_utility,
        'player 1 utility': deal.second_player_utility,
    }
    for deal in calls.deals
])

,player 0 hand,player 1 hand,probability,player 0 utility,player 1 utility
0,Jc Ad,9c Kd,1/4,3,-1
1,Jc Ad,8c Qd,1/4,3,-1
2,4d 5d,9c Kd,1/4,0,2
3,4d 5d,8c Qd,1/4,0,2


## What this establishes

- Chance selects a compatible underlying private deal.
- Decision states expose only player-observable information.
- Betting changes stacks, contributions and pot exactly.
- Fold and showdown terminals use the existing hand comparison code.
- Behavioural strategies are evaluated exactly with rational probabilities and chip EV.

The next extension can insert a `chance_children` river reveal between turn and river betting rounds while preserving the public action and reveal histories.